# CUT + Residual 제약 실험 — unpaired 구조 보존 (직접 실행)

**정답(HR) 안 쓰는 완전 unpaired.** 리파이너를 `출력 = 입력 + scale·(G(입력)−입력)` 로 묶어
왜곡·환각을 억제(구조 보존)하면서 질감을 넣는다. `RES_SCALE` 로 강도 조절.

- 커널 `trellis` → Restart Kernel → Run All
- 데이터 `refine_hat` (HAT-SR→HR, train 420 / test 60, 512) — 팀장 HAT 출력 기반
- 비교 대상: SR-HAT 14.77 / CycleGAN 13.74·0.364·0.606·198.6·5.95 / CUT(잔차없음) 13.84·0.320·0.651·179.1·7.05
- ⚠️ 셀 2 학습 ≈ 2시간+ (빨리 보려면 --n_epochs 줄이기). **RES_SCALE 튜닝**이 이 실험의 핵심 (0.3/0.5/0.7 비교)


## 0. 환경 설정 + 잔차 강도

In [ ]:
import os, sys, time, glob
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
ROOT = os.path.expanduser("~/erang_sr")
CUT  = os.path.join(ROOT, "refiners/cut_src")
os.chdir(CUT)
sys.path.insert(0, CUT)
for p in (ROOT, os.path.join(ROOT, "team_aiduo")):
    if p not in sys.path: sys.path.append(p)
DEV = "cuda"

RES_SCALE = 0.5   # ★ 잔차 강도: 1=일반CUT, 작을수록 구조보존↑ 질감↓ (0.3/0.5/0.7 비교해보기)

def load01(p):
    im = np.asarray(Image.open(p).convert("RGB")).astype(np.float32)/255.0
    return torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).float()
def save01(t, p):
    a = (t.detach().cpu().clamp(0,1).numpy().transpose(1,2,0)*255).round().astype("uint8")
    Image.fromarray(a).save(p)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "| RES_SCALE:", RES_SCALE)


## 1. 데이터 확인 (HAT-SR → HR)

In [ ]:
DR = os.path.join(ROOT, "refiners/data/refine_hat")
for s in ["trainA","trainB","testA","testB"]:
    print(f"  {s}: {len(os.listdir(os.path.join(DR,s)))}장")
a = sorted(glob.glob(DR+"/testA/*.png"))[0]; b = a.replace("testA","testB")
fig, ax = plt.subplots(1,2,figsize=(8,4))
ax[0].imshow(Image.open(a)); ax[0].set_title("HAT-SR (input)"); ax[0].axis("off")
ax[1].imshow(Image.open(b)); ax[1].set_title("HR (GT)"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## 2. CUT+Residual 학습 (unpaired, loss 관찰)
`--model cutres --res_scale RES_SCALE`. NCE가 일반 CUT보다 낮게 시작하면 잔차가 구조를 잡고 있다는 신호.

In [ ]:
sys.argv = ["train.py", "--dataroot", DR, "--name", "cut_res", "--model", "cutres", "--CUT_mode", "CUT",
    "--res_scale", str(RES_SCALE),
    "--display_id", "0", "--gpu_ids", "0", "--batch_size", "1",
    "--n_epochs", "25", "--n_epochs_decay", "10",     # <- 빨리 보려면 줄이기(예: 5,0)
    "--load_size", "512", "--crop_size", "256", "--print_freq", "200"]
from options.train_options import TrainOptions
from data import create_dataset
from models import create_model
opt = TrainOptions().parse(); opt.num_threads = 0
dataset = create_dataset(opt); model = create_model(opt)
print(f"\n학습 {len(dataset)}장 · {opt.n_epochs+opt.n_epochs_decay} epoch (res_scale={RES_SCALE})\n")
t0 = time.time(); step = 0
for epoch in range(opt.epoch_count, opt.n_epochs + opt.n_epochs_decay + 1):
    for i, data in enumerate(dataset):
        if epoch == opt.epoch_count and i == 0:
            model.data_dependent_initialize(data); model.setup(opt); model.parallelize()
        model.set_input(data); model.optimize_parameters(); step += 1
        if step % 50 == 0:
            L = model.get_current_losses()
            print(f"ep{epoch} step{step:>5}  G_GAN {L['G_GAN']:.3f}  NCE {L['NCE']:.3f}  "
                  f"D_real {L['D_real']:.3f}  ({time.time()-t0:.0f}s)")
    print(f"--- epoch {epoch} done ({time.time()-t0:.0f}s) ---")
print(f"\n학습 종료 ({time.time()-t0:.0f}s)")


## 3. refined 결과 (잔차 적용, 테스트 3장)

In [ ]:
G_cut = model.netG; G_cut.eval()
def refine_res(G, x, s):
    # 학습과 동일한 잔차 공식: real + s*(G(real)-real)
    r = x*2-1
    with torch.no_grad():
        g = G(r)
        out = (r + s*(g - r)).clamp(-1,1)
    return ((out+1)/2).clamp(0,1)

tests = sorted(glob.glob(DR+"/testA/*.png"))[:3]
fig, ax = plt.subplots(len(tests), 3, figsize=(11, 3.6*len(tests)))
ax = ax.reshape(len(tests), 3)
for rr, ta in enumerate(tests):
    x = load01(ta).to(DEV); ref = refine_res(G_cut, x, RES_SCALE)
    imgs = [x[0].cpu(), ref[0].cpu(), load01(ta.replace("testA","testB"))[0]]
    for c,(im,t) in enumerate(zip(imgs, ["HAT-SR (input)",f"CUT+Res (s={RES_SCALE})","HR (GT)"])):
        ax[rr,c].imshow(im.numpy().transpose(1,2,0)); ax[rr,c].axis("off")
        if rr==0: ax[rr,c].set_title(t)
plt.tight_layout(); plt.show()


## 4. Dual-Track 평가 — SR-HAT vs CycleGAN vs CUT+Residual

In [ ]:
from cycleGen_model import load_cyclegan_model
G_cyc = load_cyclegan_model(os.path.join(ROOT,"weights/best_G_AB.pth"), device=DEV)
def refine_plain(G, x):
    with torch.no_grad(): return ((G(x*2-1)+1)/2).clamp(0,1)

allA = sorted(glob.glob(DR+"/trainA/*.png")) + sorted(glob.glob(DR+"/testA/*.png"))
test_names = set(os.path.basename(p) for p in glob.glob(DR+"/testA/*.png"))
base = os.path.join(ROOT, "runs/eval_res")
dirs = {k: os.path.join(base,k) for k in ["sr_hat","cyclegan","cut_res"]}
hr_ref = os.path.join(base,"hr_ref")
for dd in list(dirs.values())+[hr_ref]: os.makedirs(dd, exist_ok=True)
print(f"{len(allA)}장 처리 중...")
for j, ta in enumerate(allA):
    name = os.path.basename(ta); x = load01(ta).to(DEV)
    save01(x[0], os.path.join(dirs["sr_hat"], name))
    save01(refine_plain(G_cyc, x)[0], os.path.join(dirs["cyclegan"], name))
    save01(refine_res(G_cut, x, RES_SCALE)[0], os.path.join(dirs["cut_res"], name))
    save01(load01(ta.replace("/trainA/","/trainB/").replace("/testA/","/testB/"))[0],
           os.path.join(hr_ref, name))
    if (j+1) % 100 == 0: print(f"  {j+1}/{len(allA)}")
print("처리 완료 · 지표 계산...")
from skimage.metrics import structural_similarity as ssim
import lpips as L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
lp = L.LPIPS(net="alex").to(DEV); niqe = pyiqa.create_metric("niqe", device=DEV)
def psnr(a,b):
    mse=float(((a-b)**2).mean()); return 99.0 if mse==0 else 10*np.log10(1/mse)
rows = {}
for cond, dd in dirs.items():
    ps,ss,lps = [],[],[]
    for name in sorted(test_names):
        out=load01(os.path.join(dd,name)); hr=load01(os.path.join(hr_ref,name))
        a=out[0].numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(lp(out.to(DEV)*2-1, hr.to(DEV)*2-1).mean()))
    nqs=[float(niqe(os.path.join(dd,n))) for n in os.listdir(dd)]
    try: fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    except Exception as e: fid=float("nan"); print("FID err",cond,e)
    rows[cond]=(np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs))
print(f"\n===== Dual-Track (HAT 입력, res_scale={RES_SCALE}, test 60장) =====")
print(f"{'조건':10s} | {'PSNR up':>8} {'SSIM up':>8} {'LPIPS dn':>9} | {'FID dn':>8} {'NIQE dn':>8}")
print("-"*62)
for k in ["sr_hat","cyclegan","cut_res"]:
    p,s,l,f,n=rows[k]; print(f"{k:10s} | {p:8.3f} {s:8.4f} {l:9.4f} | {f:8.3f} {n:8.3f}")
print("\n비교: 잔차없는 CUT 13.84/0.320/0.651/179.1/7.05")
print("기대: cut_res가 잔차없는 CUT보다 SSIM/PSNR↑(구조보존) 하면서 FID 우위 유지면 성공.")


In [ ]:
# ===== 추론 전용 잔차 스윕 (재학습 없음, 순수 다이얼 곡선) =====
# 셀 2에서 학습된 G_cut 하나로 추론 블렌드 s만 0→1 변화 → 학습분산 제거
import os, glob, numpy as np, torch, matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import lpips as _L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths

S_LIST = [0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]   # 0=입력(HAT-SR), 1=full CUT

G_cut.eval()
def refine_s(G, x, s):
    r = x*2-1
    with torch.no_grad():
        g = G(r)
        return ((r + s*(g-r)).clamp(-1,1)+1).mul(0.5).clamp(0,1)

testA = sorted(glob.glob(DR+"/testA/*.png"))            # test 60장
base = os.path.join(ROOT, "runs/eval_sweep")
hr_ref = os.path.join(base, "hr_ref"); os.makedirs(hr_ref, exist_ok=True)
for ta in testA:
    save01(load01(ta.replace("testA","testB"))[0], os.path.join(hr_ref, os.path.basename(ta)))

_lp = _L.LPIPS(net="alex").to(DEV); _niqe = pyiqa.create_metric("niqe", device=DEV)
def _psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m==0 else 10*np.log10(1/m)

rows=[]
for s in S_LIST:
    dd = os.path.join(base, f"s{int(round(s*100)):03d}"); os.makedirs(dd, exist_ok=True)
    ps,ss,lps,nqs=[],[],[],[]
    for ta in testA:
        x=load01(ta).to(DEV); out=refine_s(G_cut,x,s)
        save01(out[0], os.path.join(dd, os.path.basename(ta)))
        hr=load01(ta.replace("testA","testB"))
        a=out[0].cpu().numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(_psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(_lp(out.to(DEV)*2-1, hr.to(DEV)*2-1).mean()))
        nqs.append(float(_niqe(os.path.join(dd, os.path.basename(ta)))))
    fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    rows.append((s,np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs)))
    print(f"s={s:.1f} | PSNR {np.mean(ps):6.2f}  SSIM {np.mean(ss):.3f}  LPIPS {np.mean(lps):.3f}  FID {fid:7.2f}  NIQE {np.mean(nqs):.3f}")

print("\n  s   | PSNR up  SSIM up  LPIPS dn |  FID dn  NIQE dn")
print("-"*54)
for s,p,ss_,l,f,n in rows:
    print(f" {s:.1f}  | {p:7.2f} {ss_:8.3f} {l:9.3f} | {f:7.2f} {n:7.3f}")

# ---- 곡선 2장 ----
S=[r[0] for r in rows]; P=[r[1] for r in rows]; SS=[r[2] for r in rows]
LP=[r[3] for r in rows]; FD=[r[4] for r in rows]; NQ=[r[5] for r in rows]
fig,ax=plt.subplots(1,2,figsize=(13,5))
axr=ax[0].twinx()
ax[0].plot(S,LP,'o-',color='tab:blue',label='LPIPS↓')
ax[0].plot(S,SS,'^-',color='tab:green',label='SSIM↑')
axr.plot(S,FD,'s--',color='tab:red',label='FID↓')
ax[0].set_xlabel('residual scale s (0=input, 1=full CUT)'); ax[0].set_ylabel('LPIPS / SSIM'); axr.set_ylabel('FID')
ax[0].set_title('Metrics vs residual scale'); ax[0].grid(alpha=0.3)
h0,l0=ax[0].get_legend_handles_labels(); h1,l1=axr.get_legend_handles_labels()
ax[0].legend(h0+h1, l0+l1, loc='best')
ax[1].plot(LP,FD,'o-',color='purple')
for s,l,f in zip(S,LP,FD): ax[1].annotate(f"s={s:.1f}",(l,f),fontsize=9)
ax[1].set_xlabel('LPIPS down (perceptual closeness to GT)')
ax[1].set_ylabel('FID down (distribution realism)')
plt.tight_layout(); plt.savefig(os.path.join(ROOT,"runs/res_sweep_curve.png"),dpi=120); plt.show()
print("saved:", os.path.join(ROOT,"runs/res_sweep_curve.png"))

In [ ]:
# ===== 480장 확정 평가: 잔차 s=0.3 (현재 G_cut 그대로, CycleGAN FID 198.6과 동일 조건 비교) =====
S_EVAL = 0.3
import os, glob, numpy as np, torch
from cycleGen_model import load_cyclegan_model
G_cyc = load_cyclegan_model(os.path.join(ROOT,"weights/best_G_AB.pth"), device=DEV)
G_cut.eval()
def refine_s(G, x, s):
    r = x*2-1
    with torch.no_grad():
        g = G(r); return ((r + s*(g-r)).clamp(-1,1)+1).mul(0.5).clamp(0,1)
def refine_plain(G, x):
    with torch.no_grad(): return ((G(x*2-1)+1)/2).clamp(0,1)

allA = sorted(glob.glob(DR+"/trainA/*.png")) + sorted(glob.glob(DR+"/testA/*.png"))
test_names = set(os.path.basename(p) for p in glob.glob(DR+"/testA/*.png"))
base = os.path.join(ROOT, "runs/eval_res03")
dirs = {k: os.path.join(base,k) for k in ["sr_hat","cyclegan","cut_res03"]}
hr_ref = os.path.join(base,"hr_ref")
for dd in list(dirs.values())+[hr_ref]: os.makedirs(dd, exist_ok=True)
print(f"{len(allA)}장 처리 중 (s={S_EVAL})...")
for j, ta in enumerate(allA):
    name=os.path.basename(ta); x=load01(ta).to(DEV)
    save01(x[0], os.path.join(dirs["sr_hat"], name))
    save01(refine_plain(G_cyc, x)[0], os.path.join(dirs["cyclegan"], name))
    save01(refine_s(G_cut, x, S_EVAL)[0], os.path.join(dirs["cut_res03"], name))
    save01(load01(ta.replace("/trainA/","/trainB/").replace("/testA/","/testB/"))[0], os.path.join(hr_ref, name))
    if (j+1)%100==0: print(f"  {j+1}/{len(allA)}")
print("처리 완료 · 지표 계산...")
from skimage.metrics import structural_similarity as ssim
import lpips as _L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
lp=_L.LPIPS(net="alex").to(DEV); niqe=pyiqa.create_metric("niqe", device=DEV)
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m==0 else 10*np.log10(1/m)
rows={}
for cond,dd in dirs.items():
    ps,ss,lps=[],[],[]
    for name in sorted(test_names):
        out=load01(os.path.join(dd,name)); hr=load01(os.path.join(hr_ref,name))
        a=out[0].numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(lp(out.to(DEV)*2-1, hr.to(DEV)*2-1).mean()))
    nqs=[float(niqe(os.path.join(dd,n))) for n in os.listdir(dd)]
    fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    rows[cond]=(np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs))
print(f"\n===== 480장 확정 (HAT 입력, 잔차 s={S_EVAL}) =====")
print(f"{'조건':12s} | {'PSNR up':>8} {'SSIM up':>8} {'LPIPS dn':>9} | {'FID dn':>8} {'NIQE dn':>8}")
print("-"*64)
for k in ["sr_hat","cyclegan","cut_res03"]:
    p,s,l,f,n=rows[k]; print(f"{k:12s} | {p:8.3f} {s:8.4f} {l:9.4f} | {f:8.3f} {n:8.3f}")
print("\n판정 기준: cut_res03의 FID < cyclegan 198.6 이면 → 동일 480장에서 win 확정.")

In [ ]:
# ===== 견고성 검증: plain CUT(cut_hat) 모델로 추론 스윕 =====
import copy, os, glob, numpy as np, torch
# G_cut과 동일 아키텍처(둘 다 CUTModel netG)에 cut_hat 가중치 로드
G_plain = copy.deepcopy(G_cut)
sd = torch.load(os.path.join(CUT, "checkpoints/cut_hat/latest_net_G.pth"), map_location=DEV)
(G_plain.module if hasattr(G_plain, "module") else G_plain).load_state_dict(sd)
G_plain.eval()
print("plain CUT(cut_hat) 로드 완료")

def refine_s(G, x, s):
    r = x*2-1
    with torch.no_grad():
        g = G(r); return ((r + s*(g-r)).clamp(-1,1)+1).mul(0.5).clamp(0,1)

S_LIST = [0.0, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
testA = sorted(glob.glob(DR+"/testA/*.png"))
base = os.path.join(ROOT, "runs/eval_sweep_plain")
hr_ref = os.path.join(base,"hr_ref"); os.makedirs(hr_ref, exist_ok=True)
for ta in testA: save01(load01(ta.replace("testA","testB"))[0], os.path.join(hr_ref, os.path.basename(ta)))

from skimage.metrics import structural_similarity as ssim
import lpips as _L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
_lp=_L.LPIPS(net="alex").to(DEV); _niqe=pyiqa.create_metric("niqe", device=DEV)
def _psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m==0 else 10*np.log10(1/m)

rows=[]
for s in S_LIST:
    dd=os.path.join(base,f"s{int(round(s*100)):03d}"); os.makedirs(dd,exist_ok=True)
    ps,ss,lps,nqs=[],[],[],[]
    for ta in testA:
        x=load01(ta).to(DEV); out=refine_s(G_plain,x,s)
        save01(out[0], os.path.join(dd, os.path.basename(ta)))
        hr=load01(ta.replace("testA","testB"))
        a=out[0].cpu().numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(_psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(_lp(out.to(DEV)*2-1, hr.to(DEV)*2-1).mean()))
        nqs.append(float(_niqe(os.path.join(dd, os.path.basename(ta)))))
    fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    rows.append((s,np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs)))
    print(f"s={s:.1f} | PSNR {np.mean(ps):6.2f}  SSIM {np.mean(ss):.3f}  LPIPS {np.mean(lps):.3f}  FID {fid:7.2f}  NIQE {np.mean(nqs):.3f}")

print("\n  s   | PSNR up  SSIM up  LPIPS dn |  FID dn  NIQE dn   (plain CUT = cut_hat)")
print("-"*62)
for s,p,ss_,l,f,n in rows:
    print(f" {s:.1f}  | {p:7.2f} {ss_:8.3f} {l:9.3f} | {f:7.2f} {n:7.3f}")
print("\n관전: s=1.0(=순수 plain CUT) 대비, 중간 s에서 LPIPS/FID/SSIM 균형이 더 좋아지면 → sweet spot 모델무관 확정.")

In [ ]:
model.save_networks("res07_final")   # 현재 커널의 cut_res(0.7) 모델 저장 → checkpoints/cut_res/res07_final_net_G.pth
print("saved:", os.path.join(CUT, "checkpoints/cut_res/res07_final_net_G.pth"))

## 해석 & 튜닝
- `cut_res`(잔차 CUT)가 **잔차없는 CUT 대비 SSIM/PSNR이 오르는지** 확인 (구조 보존 효과).
- **RES_SCALE 튜닝**: 셀 0의 값을 0.3/0.5/0.7로 바꿔 Run All 반복 → 충실도↔질감 균형점 찾기.
  - 작을수록: 구조↑(PSNR/SSIM↑) 질감↓, 클수록: 반대.
- 완전 unpaired라 기업 요건(정답 없이 개선) 충족.
